In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [2]:
# load 3 datasets and concatenate them
df1 = pd.read_csv('../data/general/news.csv')
df1['label'] = df1['label'].map({'REAL': 0, 'FAKE': 1}) # map labels to 0 and 1
df2 = pd.read_csv('../data/general/True.csv')
df2['label'] = 0 # all entries in this dataset are real news, so label is 0
df3 = pd.read_csv('../data/general/Fake.csv')
df3['label'] = 1 # all entries in this dataset are fake news, so label is 1
df = pd.concat([df1, df2, df3], ignore_index=True)
df.sample(n=5)

,index,title,text,label,subject,date
49570,NaN,BUSTED: [Video] AARP Caught Using SUBLIMINAL M...,Could someone please explain this? I honestly ...,1,left-news,"Apr 26, 2015"
956,5088.0,Democratic convention: passionate end to day o...,A stormy opening night of the Democratic conve...,0,NaN,NaN
10621,NaN,Kansas Republican wins congressional seat in s...,"KANSAS CITY, Kan (Reuters) - Kansas Republican...",0,politicsNews,"April 12, 2017"
39073,NaN,BREAKING: OBAMACARE REPEAL Vote Cancelled…Repu...,I would love to see 237 votes on the House si...,1,politics,"Mar 23, 2017"
17289,NaN,White House says doesn't expect Congress to pa...,WASHINGTON (Reuters) - The White House said on...,0,politicsNews,"February 5, 2016"


In [3]:
# drop columns that are not needed for classification
df = df.drop(columns=['index', 'title', 'subject', 'date'])

# remove entries where the text length is <200, as they are likely to be noise
df['text_length'] = df['text'].apply(len)
df = df[df['text_length'] >= 200].drop(columns=['text_length'])

In [4]:
# since we are partially using ISOT dataset, we need to perform some minor preprocessing to remove 
# the source of the news from the text, as it can be a strong signal for classification and can lead to overfitting. 
# We will remove any URLs and email addresses from the text.

def clean_text(text):
    # 1. Remove Publisher tags (ISOT specific)
    text = re.sub(r'^.*? - ', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # 3. Standard cleaning
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s!\?-]', '', text) # Keep text, spaces, !, ?, nums 0-9, and -
    return text

df['text'] = df['text'].apply(clean_text)

In [5]:
# split data into train and test sets
X, y = df['text'], df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# preprocessing function to be used in the pipeline, which will clean the text using the same function as above
preprocess = FunctionTransformer(lambda x: pd.Series(x).apply(clean_text))

# create a pipeline with TfidfVectorizer and SGDClassifier, with hyperparameter weights already tuned previously using GridSearchCV
pipeline = make_pipeline(
    preprocess,
    TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english'),
    SGDClassifier(loss='hinge', penalty=None, alpha=1e-4, random_state=42, learning_rate='pa2', eta0=1.0, class_weight={0 : 1, 1 : 2})
)

In [7]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9662066166023949
F1 Score: 0.966809528555766
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      4842
           1       0.97      0.97      0.97      5012

    accuracy                           0.97      9854
   macro avg       0.97      0.97      0.97      9854
weighted avg       0.97      0.97      0.97      9854

Confusion Matrix:
 [[4671  171]
 [ 162 4850]]


In [8]:
# The example article is from the BBC, and is a real news article. The model should ideally predict it as 0 (real news).
# Link: https://www.bbc.com/news/articles/c99jxryd9xko
with open('real_article.txt', 'r') as f:
    real_article = f.read()

In [9]:
pipeline.predict([real_article])

array([0])

Since the model predicted that the article was real, then it matches what we know about reality.

What if we try it on a fake news article?

In [10]:
# The example article is from The Gateway Pundit, and is a fake news article. The model should ideally predict it as 1 (fake news).
# I am not giving you the link to this garbage...
with open('fake_article.txt', 'r') as f:
    fake_article = f.read()

In [11]:
pipeline.predict([fake_article])

array([1])

The model marks it as fake!

In [12]:
# predicted probabilities of each article
print("Real article predicted probability of being fake news:", pipeline.decision_function([real_article]))
print("Fake article predicted probability of being fake news:", pipeline.decision_function([fake_article]))

Real article predicted probability of being fake news: [-5.05710677]
Fake article predicted probability of being fake news: [1.2099135]


It seems like the model was extremely certain that the BBC article was real, while it was lightly certain that the GatewayPundit article was actually fake. So potentially, if a fake news article was even better written than the fake article, then it could be passed off as real by the model!

In [13]:
# 20 largest factors that can predict whether an article is fake or real news, according to the model. 
# We will use the coefficients of the SGDClassifier to find the most important features.

feature_names = pipeline.named_steps['tfidfvectorizer'].get_feature_names_out()
coefficients = pipeline.named_steps['sgdclassifier'].coef_[0]
feature_importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
feature_importance['abs_coefficient'] = feature_importance['coefficient'].abs()
top_features = feature_importance.sort_values(by='abs_coefficient', ascending=False).head(20)
print(top_features[['feature', 'coefficient']])

               feature  coefficient
4311             image    15.376503
7719              said   -12.125400
2714             doesn    11.314248
6756  president donald   -10.301645
6767   president trump    10.092941
9094      told reuters    -9.977416
2568              didn     9.679349
9319            trumps    -9.449694
665                 ap     9.105883
9789               wfb     8.965945
4642               isn     8.537232
7797           saidthe     8.214761
9709              wasn     8.143789
6024               nyp     7.989212
9534                ve     7.602205
5164                ll     7.257277
730               aren     7.157808
6077           october     6.819190
3110               est    -6.703591
3061            entire     6.537417
